# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [10]:
#Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Make a copy to avoid modifying the original DataFrame directly
df = spaceship.copy()

# Separate target variable
X = df.drop('Transported', axis=1)
y = df['Transported']

In [4]:
# Feature Engineering for 'Cabin' column
# Split 'Cabin' into 'Cabin_deck', 'Cabin_num', 'Cabin_side'
X[['Cabin_deck', 'Cabin_num', 'Cabin_side']] = X['Cabin'].str.split('/', expand=True)
X = X.drop('Cabin', axis=1)

# Convert 'Cabin_num' to numeric, coerce errors to NaN
X['Cabin_num'] = pd.to_numeric(X['Cabin_num'], errors='coerce')

In [6]:
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

# Remove 'PassengerId' and 'Name' from features if they are present
if 'PassengerId' in numerical_features:
    numerical_features.remove('PassengerId')
if 'PassengerId' in categorical_features:
    categorical_features.remove('PassengerId')
if 'Name' in categorical_features:
    categorical_features.remove('Name')

# Define preprocessing steps
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough' # Keep other columns (like PassengerId for submission if needed)
)

# Apply preprocessing
X_preprocessed = preprocessor.fit_transform(X)

# Get feature names after one-hot encoding for categorical features and including passthrough columns
all_feature_names = preprocessor.get_feature_names_out()

# Convert the preprocessed data back to a DataFrame
X_processed_df = pd.DataFrame(X_preprocessed, columns=all_feature_names)

print("Shape of preprocessed data:", X_processed_df.shape)
display(X_processed_df.head())

Shape of preprocessed data: (8693, 29)


,num__Age,num__RoomService,num__FoodCourt,num__ShoppingMall,num__Spa,num__VRDeck,num__Cabin_num,cat__HomePlanet_Earth,cat__HomePlanet_Europa,cat__HomePlanet_Mars,...,cat__Cabin_deck_C,cat__Cabin_deck_D,cat__Cabin_deck_E,cat__Cabin_deck_F,cat__Cabin_deck_G,cat__Cabin_deck_T,cat__Cabin_side_P,cat__Cabin_side_S,remainder__PassengerId,remainder__Name
0,0.709437,-0.34059,-0.287314,-0.290817,-0.276663,-0.269023,-1.186627,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0001_01,Maham Ofracculy
1,-0.336717,-0.175364,-0.281669,-0.248968,0.211505,-0.230194,-1.186627,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0002_01,Juanna Vines
2,2.034566,-0.275409,1.955616,-0.290817,5.694289,-0.225782,-1.186627,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0003_01,Altark Susent
3,0.290975,-0.34059,0.517406,0.330225,2.683471,-0.098708,-1.186627,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0003_02,Solam Susent
4,-0.894666,0.118709,-0.243409,-0.038048,0.225732,-0.267258,-1.184651,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0004_01,Willy Santantines


**Perform Train Test Split**

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X_processed_df, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (6954, 29)
X_test shape: (1739, 29)
y_train shape: (6954,)
y_test shape: (1739,)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [12]:
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=20),
                               n_estimators=100, # number of models to use
                               max_samples = 1000)

In [17]:
# Drop non-numerical 'remainder' columns from X_train before fitting
X_train_cleaned = X_train.drop(columns=['remainder__PassengerId', 'remainder__Name'], errors='ignore')

bagging_reg.fit(X_train_cleaned, y_train)

BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=20),
                 max_samples=1000, n_estimators=100)

In [18]:
print(f"Shape of X_train_cleaned: {X_train_cleaned.shape}")
print(f"Shape of y_train: {y_train.shape}")

Shape of X_train_cleaned: (6954, 27)
Shape of y_train: (6954,)


Evaluating the model performance

In [19]:
# Drop non-numerical 'remainder' columns from X_test before making predictions
X_test_cleaned = X_test.drop(columns=['remainder__PassengerId', 'remainder__Name'], errors='ignore')

y_pred = bagging_reg.predict(X_test_cleaned)

# Evaluate the model
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print(f"R2 Score: {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

R2 Score: 0.4492
Mean Absolute Error (MAE): 0.2795
Mean Squared Error (MSE): 0.1377
Root Mean Squared Error (RMSE): 0.3710


- Random Forests

In [20]:
print('\n--- Random Forest Regressor ---')

# Initialize and train the Random Forest Regressor
random_forest_reg = RandomForestRegressor(n_estimators=100, random_state=42)
random_forest_reg.fit(X_train_cleaned, y_train)

# Make predictions on the cleaned test set
y_pred_rf = random_forest_reg.predict(X_test_cleaned)

# Evaluate the Random Forest model
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)

print(f"R2 Score (Random Forest): {r2_rf:.4f}")
print(f"Mean Absolute Error (MAE) (Random Forest): {mae_rf:.4f}")
print(f"Mean Squared Error (MSE) (Random Forest): {mse_rf:.4f}")
print(f"Root Mean Squared Error (RMSE) (Random Forest): {rmse_rf:.4f}")


--- Random Forest Regressor ---
R2 Score (Random Forest): 0.4324
Mean Absolute Error (MAE) (Random Forest): 0.2627
Mean Squared Error (MSE) (Random Forest): 0.1419
Root Mean Squared Error (RMSE) (Random Forest): 0.3767


- Gradient Boosting

In [22]:
print('\n--- Gradient Boosting Regressor ---')

# Initialize and train the Gradient Boosting Regressor
gradient_boosting_reg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gradient_boosting_reg.fit(X_train_cleaned, y_train)

# Make predictions on the cleaned test set
y_pred_gb = gradient_boosting_reg.predict(X_test_cleaned)

# Evaluate the Gradient Boosting model
r2_gb = r2_score(y_test, y_pred_gb)
mae_gb = mean_absolute_error(y_test, y_pred_gb)
mse_gb = mean_squared_error(y_test, y_pred_gb)
rmse_gb = root_mean_squared_error(y_test, y_pred_gb)

print(f"R2 Score (Gradient Boosting): {r2_gb:.4f}")
print(f"Mean Absolute Error (MAE) (Gradient Boosting): {mae_gb:.4f}")
print(f"Mean Squared Error (MSE) (Gradient Boosting): {mse_gb:.4f}")
print(f"Root Mean Squared Error (RMSE) (Gradient Boosting): {rmse_gb:.4f}")


--- Gradient Boosting Regressor ---
R2 Score (Gradient Boosting): 0.4682
Mean Absolute Error (MAE) (Gradient Boosting): 0.2812
Mean Squared Error (MSE) (Gradient Boosting): 0.1329
Root Mean Squared Error (RMSE) (Gradient Boosting): 0.3646


### Additional: Ajuste de Hiperparámetros para Gradient Boosting

Para optimizar el rendimiento del `GradientBoostingRegressor`, podemos realizar una búsqueda de hiperparámetros. `GridSearchCV` es una herramienta útil que prueba todas las combinaciones de un conjunto predefinido de hiperparámetros y selecciona la que ofrece el mejor rendimiento según una métrica de evaluación específica (por ejemplo, R2, MSE).

Los hiperparámetros clave a considerar para `GradientBoostingRegressor` incluyen:
- `n_estimators`: El número de etapas de boosting a realizar. Un número mayor puede llevar a un mejor ajuste, pero también a un sobreajuste.
- `learning_rate`: Contribuye a la contracción de cada árbol. Un valor más pequeño requiere un mayor número de estimadores, pero puede mejorar la generalización.
- `max_depth`: La profundidad máxima de los estimadores individuales del árbol de regresión. Controla la complejidad de cada árbol.

In [23]:
from sklearn.model_selection import GridSearchCV

print('\n--- Ajuste de Hiperparámetros para Gradient Boosting ---')

# Definir el rango de hiperparámetros a buscar
param_grid = {
    'n_estimators': [50, 100, 200], # Número de árboles
    'learning_rate': [0.01, 0.1, 0.2], # Tasa de aprendizaje
    'max_depth': [3, 4, 5] # Profundidad máxima de cada árbol
}

# Inicializar el modelo Gradient Boosting
gradient_boosting_reg = GradientBoostingRegressor(random_state=42)

# Configurar GridSearchCV
# cv=5 indica 5-fold cross-validation
# scoring='r2' para optimizar la métrica R2
grid_search = GridSearchCV(estimator=gradient_boosting_reg, param_grid=param_grid,
                           cv=5, scoring='r2', n_jobs=-1, verbose=1)

# Ejecutar la búsqueda de la cuadrícula en los datos de entrenamiento limpios
grid_search.fit(X_train_cleaned, y_train)

print(f"Mejores hiperparámetros encontrados: {grid_search.best_params_}")
print(f"Mejor puntuación R2 (en validación cruzada): {grid_search.best_score_:.4f}")

# Obtener el mejor modelo
best_gb_model = grid_search.best_estimator_

# Realizar predicciones con el mejor modelo en el conjunto de prueba
y_pred_gb_tuned = best_gb_model.predict(X_test_cleaned)

# Evaluar el mejor modelo
r2_gb_tuned = r2_score(y_test, y_pred_gb_tuned)
mae_gb_tuned = mean_absolute_error(y_test, y_pred_gb_tuned)
mse_gb_tuned = mean_squared_error(y_test, y_pred_gb_tuned)
rmse_gb_tuned = root_mean_squared_error(y_test, y_pred_gb_tuned)

print(f"\n--- Evaluación del Mejor Modelo Gradient Boosting ---")
print(f"R2 Score (Gradient Boosting Ajustado): {r2_gb_tuned:.4f}")
print(f"Mean Absolute Error (MAE) (Gradient Boosting Ajustado): {mae_gb_tuned:.4f}")
print(f"Mean Squared Error (MSE) (Gradient Boosting Ajustado): {mse_gb_tuned:.4f}")
print(f"Root Mean Squared Error (RMSE) (Gradient Boosting Ajustado): {rmse_gb_tuned:.4f}")


--- Ajuste de Hiperparámetros para Gradient Boosting ---
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Mejores hiperparámetros encontrados: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100}
Mejor puntuación R2 (en validación cruzada): 0.4814

--- Evaluación del Mejor Modelo Gradient Boosting ---
R2 Score (Gradient Boosting Ajustado): 0.4803
Mean Absolute Error (MAE) (Gradient Boosting Ajustado): 0.2712
Mean Squared Error (MSE) (Gradient Boosting Ajustado): 0.1299
Root Mean Squared Error (RMSE) (Gradient Boosting Ajustado): 0.3604


- Adaptive Boosting

In [24]:
gb_reg = GradientBoostingRegressor(max_depth=20,
                                   n_estimators=100)

In [25]:
gb_reg.fit(X_train_cleaned, y_train)

GradientBoostingRegressor(max_depth=20)

In [29]:
y_pred_test_gb = gb_reg.predict(X_test_cleaned)

print(f"MAE, {mean_absolute_error(y_pred_test_gb, y_test): .2f}")
print(f"MSE, {mean_squared_error(y_pred_test_gb, y_test): .2f}")
print(f"RMSE, {root_mean_squared_error(y_pred_test_gb, y_test): .2f}")
print(f"R2 score, {gb_reg.score(X_test_cleaned, y_test): .2f}")

MAE,  0.27
MSE,  0.21
RMSE,  0.46
R2 score,  0.16


Comparison

In [30]:
import pandas as pd

# Métricas para Bagging Regressor (sin ajustar)
r2_bagging = r2
mae_bagging = mae
mse_bagging = mse
rmse_bagging = rmse

# Métricas para Random Forest Regressor (sin ajustar)
r2_rf_model = r2_rf
mae_rf_model = mae_rf
mse_rf_model = mse_rf
rmse_rf_model = rmse_rf

# Métricas para Gradient Boosting Regressor (ajustado)
r2_gb_tuned_model = r2_gb_tuned
mae_gb_tuned_model = mae_gb_tuned
mse_gb_tuned_model = mse_gb_tuned
rmse_gb_tuned_model = rmse_gb_tuned

# Crear un DataFrame para la comparación
comparison_df = pd.DataFrame({
    'Modelo': ['Bagging Regressor', 'Random Forest Regressor', 'Gradient Boosting Regressor (Ajustado)']
})

comparison_df['R2 Score'] = [r2_bagging, r2_rf_model, r2_gb_tuned_model]
comparison_df['MAE'] = [mae_bagging, mae_rf_model, mae_gb_tuned_model]
comparison_df['MSE'] = [mse_bagging, mse_rf_model, mse_gb_tuned_model]
comparison_df['RMSE'] = [rmse_bagging, rmse_rf_model, rmse_gb_tuned_model]

# Redondear las métricas para una mejor lectura
comparison_df = comparison_df.round(4)

display(comparison_df)

print('\n--- Análisis ---')
print("Observando los resultados:")
print("- Un R2 Score más alto indica que el modelo explica mejor la varianza de la variable objetivo.")
print("- Un MAE, MSE y RMSE más bajos indican un menor error de predicción del modelo.")
print("\nBasado en estas métricas, podemos identificar el modelo con el mejor rendimiento general.")

,Modelo,R2 Score,MAE,MSE,RMSE
0,Bagging Regressor,0.4492,0.2795,0.1377,0.3710
1,Random Forest Regressor,0.4324,0.2627,0.1419,0.3767
2,Gradient Boosting Regressor (Ajustado),0.4803,0.2712,0.1299,0.3604



--- Análisis ---
Observando los resultados:
- Un R2 Score más alto indica que el modelo explica mejor la varianza de la variable objetivo.
- Un MAE, MSE y RMSE más bajos indican un menor error de predicción del modelo.

Basado en estas métricas, podemos identificar el modelo con el mejor rendimiento general.


Which model is the best and why?

Gradient Boosting Regressor (Ajustado)	because of its R2=0.4803